# ML Agent vs Market Benchmarks

Compares the trained RL agent — **rolling-window (walk-forward) model**, continuous out-of-sample 2010→2026 — against:
- **BOVA11** — Bovespa ETF (IBOV index proxy), the passive-market benchmark
- **Individual stocks** — VALE3, PETR4, WEGE3, ITUB4 (buy-and-hold each)
- **SELIC / CDI** — risk-free / interbank rate (fixed-income benchmark)
- **IPCA** — inflation, to show real (purchasing-power) returns

Everything is normalized to R$100k starting capital on the first test date.

**Rate semantics (BCB SGS):** SELIC (series 11) and CDI (series 12) are stored as *daily* rates in percent; IPCA (series 433) is *monthly* percent inflation. Handled correctly below.

**Prereq:** run `python -m src.agent.rolling_eval` first to produce `data/backtest/walkforward_results.parquet`.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go

ROOT = Path.cwd()
while not (ROOT / 'data' / 'backtest').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent  # works whether run from repo root or src/visualizations/

INITIAL_CAPITAL = 100_000

# Agent = rolling-window (walk-forward) model: continuous OOS curve stitched by
# rolling_eval.py. Not the single fixed-split model in results.parquet.
BT = ROOT / 'data/backtest'
res_path = BT / 'walkforward_results.parquet'
if not res_path.exists():
    raise FileNotFoundError(
        f'{res_path} not found — run `python -m src.agent.rolling_eval` first.')
print(f'Source: {res_path.name} (rolling-window model)')

# Agent backtest (one value column per strategy, written by rolling_eval.py)
backtest = pd.read_parquet(res_path)
backtest['date'] = pd.to_datetime(backtest['date'])
dates = backtest['date']
test_start = dates.min()

# Split-adjusted prices for index + individual stocks
prices = pd.read_parquet(ROOT / 'data/raw/prices', columns=['ticker', 'trade_date', 'adj_close'])
prices['trade_date'] = pd.to_datetime(prices['trade_date'])

# Macro series (BCB): column is 'reference_date' + the series name
macro = {}
for name in ['selic', 'cdi', 'ipca']:
    p = ROOT / f'data/raw/macro/{name}.parquet'
    if p.exists():
        s = pd.read_parquet(p)
        s['reference_date'] = pd.to_datetime(s['reference_date'])
        macro[name] = s.set_index('reference_date')[name].sort_index()

print(f'Test period: {test_start.date()} → {dates.max().date()}  ({len(backtest)} trading days)')
print(f'Strategies : {[c.removeprefix("value_") for c in backtest.columns if c.startswith("value_")]}')
print(f'Macro      : {list(macro)}')

Source: walkforward_results.parquet (rolling-window model)
Test period: 2010-01-05 → 2026-01-09  (3965 trading days)
Strategies : ['agent', 'equal_weight', 'market_cap', 'inv_vol']
Macro      : ['selic', 'cdi', 'ipca']


In [2]:
# --- helpers: normalize each benchmark to an R$100k value series on the agent's dates ---

def price_curve(ticker):
    """Buy-and-hold value of `ticker`, normalized to INITIAL_CAPITAL at test_start."""
    px = (prices[prices['ticker'] == ticker]
          .set_index('trade_date')['adj_close'].sort_index())
    px = px[px.index >= test_start]
    if px.empty:
        return None
    px = px.reindex(dates).ffill().bfill()
    return px / px.iloc[0] * INITIAL_CAPITAL

def daily_rate_curve(name):
    """Cumulative value of a daily-rate series (SELIC/CDI: percent per day)."""
    r = macro[name].reindex(dates).ffill().bfill().to_numpy() / 100.0
    return pd.Series((1 + r).cumprod() * INITIAL_CAPITAL, index=dates.values)

def inflation_curve():
    """Cumulative IPCA (monthly percent) compounded, ffilled to daily, base=INITIAL_CAPITAL."""
    ipca = macro['ipca']
    ipca = ipca[ipca.index >= test_start - pd.Timedelta(days=40)]  # one month of lead-in
    cum = (1 + ipca / 100).cumprod()
    daily = cum.reindex(dates, method='ffill')
    return pd.Series((daily / daily.iloc[0] * INITIAL_CAPITAL).values, index=dates.values)

agent = pd.Series(backtest['value_agent'].values, index=dates.values)
print('helpers ready')

helpers ready


## 1. Agent vs BOVA11 & Individual Stocks

Log scale, all starting at R$100k. BOVA11 is the passive-index baseline; the four stocks are single-name buy-and-hold. (PETR4's raw run-up is dividend-driven — adj_close reinvests them.)

In [3]:
series = {'agent': pd.Series(agent.values, index=dates)}
for t in ['BOVA11', 'VALE3', 'PETR4', 'WEGE3', 'ITUB4']:
    c = price_curve(t)
    if c is not None:
        series[t] = c

curves = pd.DataFrame(series)

fig = go.Figure()
for col in curves.columns:
    fig.add_trace(go.Scatter(
        x=curves.index, y=curves[col], name=col,
        line=dict(width=3 if col == 'agent' else 1.6,
                  dash='dash' if col == 'BOVA11' else None)))
fig.update_layout(title='Agent vs BOVA11 & Big Names (R$100k start, log)',
                  yaxis_type='log', yaxis_title='Value (R$)', height=550)
fig.show()

print((curves.iloc[-1] / INITIAL_CAPITAL - 1).sort_values(ascending=False)
      .map('{:+.1%}'.format).to_string())

agent     +3989.5%
WEGE3     +1773.1%
PETR4     +1280.3%
ITUB4      +570.7%
VALE3      +295.1%
BOVA11     +129.0%


## 2. Agent vs Fixed Income (SELIC / CDI)

SELIC is the risk-free rate; CDI tracks it closely. The gap between the agent and SELIC is the excess return earned for taking equity risk.

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=dates, y=agent.values, name='Agent', line=dict(width=3)))
bova = price_curve('BOVA11')
if bova is not None:
    fig.add_trace(go.Scatter(x=dates, y=bova.values, name='BOVA11', line=dict(width=2, dash='dot')))
for name in ['selic', 'cdi']:
    if name in macro:
        fig.add_trace(go.Scatter(x=dates, y=daily_rate_curve(name).values,
                                 name=name.upper(), line=dict(width=2, dash='dash')))
fig.update_layout(title='Agent vs BOVA11, Risk-Free (SELIC) & CDI',
                  yaxis_type='log', yaxis_title='Value (R$)', height=450)
fig.show()

if 'selic' in macro:
    selic_val = daily_rate_curve('selic')
    n_yrs = len(dates) / 252
    agent_cagr = (agent.iloc[-1] / INITIAL_CAPITAL) ** (1 / n_yrs) - 1
    selic_cagr = (selic_val.iloc[-1] / INITIAL_CAPITAL) ** (1 / n_yrs) - 1
    print(f'Agent CAGR : {agent_cagr:+.1%}')
    print(f'SELIC CAGR : {selic_cagr:+.1%}')
    print(f'Excess (agent - SELIC): {agent_cagr - selic_cagr:+.1%}/yr')

## 3. Real (Inflation-Adjusted) Return vs IPCA

Nominal gains overstate wealth in a high-inflation economy. Dividing the agent's value by cumulative IPCA gives real purchasing power.

In [5]:
if 'ipca' in macro:
    ipca_val = inflation_curve()
    real_agent = agent / (ipca_val / INITIAL_CAPITAL)

    fig = go.Figure()
    fig.add_trace(go.Scatter(x=dates, y=agent.values, name='Agent (nominal)', line=dict(width=3)))
    fig.add_trace(go.Scatter(x=dates, y=real_agent.values, name='Agent (real, ex-IPCA)',
                             line=dict(width=3, dash='dash')))
    fig.add_trace(go.Scatter(x=dates, y=ipca_val.values, name='IPCA (cumulative inflation)',
                             line=dict(width=2, dash='dot')))
    fig.update_layout(title='Agent: Nominal vs Real Return (IPCA-adjusted)',
                      yaxis_type='log', yaxis_title='Value (R$)', height=450)
    fig.show()

    print(f"Nominal return : {agent.iloc[-1] / INITIAL_CAPITAL - 1:+.1%}")
    print(f"Real return    : {real_agent.iloc[-1] / INITIAL_CAPITAL - 1:+.1%}")
    print(f"Inflation      : {ipca_val.iloc[-1] / INITIAL_CAPITAL - 1:+.1%}")
else:
    print('IPCA not available')

Nominal return : +3989.5%
Real return    : +1573.8%
Inflation      : +144.3%


## 4. Summary Table

Total return, annualized return, Sharpe, and max drawdown for the agent and every equity benchmark (fixed income excluded — near-zero volatility makes Sharpe meaningless).

In [6]:
import sys
sys.path.insert(0, str(ROOT))
from src.agent.metrics import compute_all

rows = {}
for name, curve in curves.items():
    rets = np.log(curve / curve.shift(1)).fillna(0.0).to_numpy()
    rows[name] = compute_all(rets, curve.to_numpy())

table = pd.DataFrame(rows).T[['cumulative_return', 'annualized_return', 'sharpe', 'max_drawdown']]
fmt = table.copy()
for col in ['cumulative_return', 'annualized_return', 'max_drawdown']:
    fmt[col] = fmt[col].map('{:+.1%}'.format)
fmt['sharpe'] = table['sharpe'].round(3)
print(fmt.to_string())

if 'BOVA11' in rows:
    print(f"\nAgent excess Sharpe vs BOVA11: {rows['agent']['sharpe'] - rows['BOVA11']['sharpe']:+.3f}")

       cumulative_return annualized_return  sharpe max_drawdown
agent           +3945.6%            +26.5%   1.113       +47.9%
BOVA11           +129.0%             +5.4%   0.222       +49.7%
VALE3            +295.1%             +9.1%   0.224       +81.3%
PETR4           +1280.3%            +18.2%   0.369       +85.2%
WEGE3           +1773.1%            +20.5%   0.594       +49.8%
ITUB4            +570.7%            +12.9%   0.400       +42.1%

Agent excess Sharpe vs BOVA11: +0.891


## 5. Agent Concentration — Top Average Holdings

Which names the agent actually leans on, on average across the test period.

In [7]:
weight_cols = [c for c in backtest.columns if c.startswith('w_')]
avg = (backtest[weight_cols].mean() * 100).sort_values(ascending=False).head(15)
avg.index = avg.index.str.removeprefix('w_')

fig = go.Figure(go.Bar(x=avg.values, y=avg.index, orientation='h'))
fig.update_layout(title='Agent — Top 15 Average Holdings (%)',
                  xaxis_title='Mean weight (%)', height=450,
                  yaxis=dict(autorange='reversed'))
fig.show()

print(f'Top-10 mean weight: {avg.head(10).sum():.1f}%  (100% = fully concentrated in 10 names)')

Top-10 mean weight: 10.0%  (100% = fully concentrated in 10 names)


## How to read this

- **vs BOVA11:** agent Sharpe > BOVA11 Sharpe ⇒ the policy beat passive indexing on a risk-adjusted basis.
- **vs SELIC:** positive excess CAGR is the reward for bearing equity risk (in Brazil SELIC is a high bar — ~11%/yr).
- **Real return:** what survives inflation; the number that matters for actual purchasing power.
- **Concentration:** high top-10 weight + outperformance = genuine stock selection; + underperformance = concentrated into losers.

**Note:** these numbers reflect the rolling-window model in `walkforward_results.parquet`. Re-run `python -m src.agent.rolling_eval` after retraining to refresh.